In [ ]:
import pandas as pd

# Read the CSV file into a DataFrame named 'forrest'
train = pd.read_csv('/kaggle/input/forest-cover-type-prediction/train.csv')

# Display the first few rows of the DataFrame to verify the data has been loaded
print(train.head())

In [ ]:
train.info()

In [ ]:
#check if there is any class imbalancing
train['Cover_Type'].value_counts(normalize= True)

In [ ]:
import numpy as np
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
import matplotlib.pyplot as plt

scaler = StandardScaler()

# Extract features and target variable
X = train.drop(['Id', 'Cover_Type'], axis=1)
y = train['Cover_Type']

In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from keras.utils import to_categorical
from sklearn.preprocessing import StandardScaler

# Number of folds for cross-validation
num_folds = 5
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

# Initialize lists to store the results
accuracy_per_fold_xgboost = []

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    print(f"Training XGBoost on Fold {fold + 1}...")

    # Split the data into training and validation sets
    X_fold_train, X_fold_val = X.iloc[train_index], X.iloc[val_index]
    y_fold_train, y_fold_val = y.iloc[train_index], y.iloc[val_index]
    
    y_fold_train_encoded = to_categorical(y_fold_train - 1)  # Subtracting 1 to convert class labels from 1 to 7 to 0 to 6
    y_fold_val_encoded = to_categorical(y_fold_val - 1)  # Subtracting 1 to convert class labels from 1 to 7 to 0 to 6

     # Scale the data using the previously fitted scaler
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)  # Use transform, not fit_transform
    
    # Create and train the XGBoost model
    xgb_model = XGBClassifier(n_estimators=100, random_state=42)  # You can adjust n_estimators and other parameters
    xgb_model.fit(X_fold_train_scaled, y_fold_train_encoded)

    # Make predictions on the validation set
    y_pred = xgb_model.predict(X_fold_val_scaled)

    # Evaluate accuracy for this fold
    accuracy = accuracy_score(y_fold_val_encoded, y_pred)
    accuracy_per_fold_xgboost.append(accuracy)

# Print the average performance across all folds
print("\nAverage Validation Accuracy (XGBoost): ", np.mean(accuracy_per_fold_xgboost))


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score
from keras.utils import to_categorical

# Number of folds for cross-validation
num_folds = 10
kf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=42)

# Initialize lists to store the results
accuracy_per_fold_xgboost = []

# Iterate through each fold
for fold, (train_index, val_index) in enumerate(kf.split(X, y)):
    print(f"Training XGBoost on Fold {fold + 1}...")

    # Split the data into training and validation sets
    X_fold_train, X_fold_val = X.iloc[train_index], X.iloc[val_index]
    y_fold_train, y_fold_val = y.iloc[train_index], y.iloc[val_index]
    
    y_fold_train_encoded = to_categorical(y_fold_train - 1)  # Subtracting 1 to convert class labels from 1 to 7 to 0 to 6
    y_fold_val_encoded = to_categorical(y_fold_val - 1)  # Subtracting 1 to convert class labels from 1 to 7 to 0 to 6

     # Scale the data using the previously fitted scaler
    X_fold_train_scaled = scaler.fit_transform(X_fold_train)
    X_fold_val_scaled = scaler.transform(X_fold_val)  # Use transform, not fit_transform
    
    # Create and train the XGBoost model
    xgb_model = XGBClassifier(n_estimators=100, random_state=42)  # You can adjust n_estimators and other parameters
    xgb_model.fit(X_fold_train_scaled, y_fold_train_encoded)

    # Make predictions on the validation set
    y_pred = xgb_model.predict(X_fold_val_scaled)

    # Evaluate accuracy for this fold
    accuracy = accuracy_score(y_fold_val_encoded, y_pred)
    accuracy_per_fold_xgboost.append(accuracy)

# Print the average performance across all folds
print("\nAverage Validation Accuracy (XGBoost): ", np.mean(accuracy_per_fold_xgboost))


In [ ]:
# Plot training & validation accuracy values for the last fold
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.plot(accuracy_per_fold_xgboost, label='Validation Accuracy')
plt.title('Model accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.show()

**Submission**

In [ ]:
import pandas as pd

# Load the test set into a DataFrame
test_set = pd.read_csv('/kaggle/input/forest-cover-type-prediction/test.csv')

# Display the first few rows of the test set
print(test_set.head())

In [ ]:
X_test = test_set.drop(['Id'], axis=1)
# Apply the same scaling to the test set features
X_test_scaled = scaler.transform(X_test.values)

# Make predictions
predictions = xgb_model.predict(X_test_scaled)

# Assuming you have a classification task and want to get the predicted class labels
predicted_labels = np.argmax(predictions, axis=1) + 1  # Add 1 to convert back to class labels starting from 1

# Create a DataFrame with the observation Id and predicted cover type
submission_df = pd.DataFrame({'Id': test_set['Id'], 'Cover_Type': predicted_labels})

print(submission_df)

# Save the DataFrame to a CSV file
submission_df.to_csv('submission_file.csv', index=False)